# Lab 3: Output Guardrails & Content Safety with LLM Guard

## Learning Objectives
By the end of this lab, you will:
- Use `llm-guard`'s `Toxicity` and `NoRefusal` output scanners
- Use `Regex` scanner for custom PII blocking
- Use `Relevance` to check if output is relevant to context
- Build a full `SecurityLayer` pipeline using `litellm`

In [ ]:
!uv pip install litellm llm-guard -q

In [ ]:
import logging
import os
from llm_guard import scan_output
from llm_guard.output_scanners import Toxicity, NoRefusal, Regex, Relevance
from litellm import completion

logging.basicConfig(level=logging.INFO)
print('Imports OK')

---
## Part 1: Toxicity and Refusals

Check if the model generated toxic content or refused to answer unhelpfully.

In [ ]:
toxicity_scanner = Toxicity(threshold=0.5)
refusal_scanner = NoRefusal(threshold=0.5)

outputs = [
    "The transformer uses multi-head attention.",
    "I will not answer that question.",
    "You are a terrible person."
]

for out in outputs:
    # scan_output signature expects prompt as well for context
    _, valid_tox, risk_tox = toxicity_scanner.scan("", out)
    _, valid_ref, risk_ref = refusal_scanner.scan("", out)
    
    print(f"Output: {out[:50]}")
    print(f"  Toxicity Valid: {valid_tox} | Risk: {risk_tox}")
    print(f"  NoRefusal Valid: {valid_ref} | Risk: {risk_ref}")
    print()

---
## Part 2: Regex for PII

Ensure no sensitive patterns (like SSNs or emails) are leaked.

In [ ]:
regex_scanner = Regex(patterns=[r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"], is_blocked=True)

test_out = "Contact me at admin@secret.com for details."
sanitized, is_valid, risk = regex_scanner.scan("", test_out)

print(f"Valid: {is_valid} | Risk: {risk}")
print(f"Sanitized Output: {sanitized}")

---
## Part 3: Relevance (Hallucination Defense)

Check if the output actually answers the prompt/context.

In [ ]:
relevance_scanner = Relevance(threshold=0.5)

prompt = "What is the capital of France?"
response_good = "The capital of France is Paris."
response_bad = "The transformer model was introduced in 2017."

_, v1, r1 = relevance_scanner.scan(prompt, response_good)
_, v2, r2 = relevance_scanner.scan(prompt, response_bad)

print("Good response valid:", v1, "| Risk:", r1)
print("Bad response valid:", v2, "| Risk:", r2)

---
## Part 4: Unified Output Guard

Combine them using `scan_output`.

In [ ]:
out_scanners = [
    Toxicity(threshold=0.5),
    NoRefusal(threshold=0.5),
    Relevance(threshold=0.5)
]

def process_output(prompt: str, output: str):
    sanitized_output, results_valid, results_score = scan_output(out_scanners, prompt, output)
    
    if any(not is_valid for is_valid in results_valid.values()):
        failed_scanners = [name for name, is_valid in results_valid.items() if not is_valid]
        return False, f"Blocked by: {failed_scanners}"
        
    return True, sanitized_output

print("Setup complete.")

---
## Part 5: Full Pipeline Integration with LiteLLM

Let's generate a response using LiteLLM and run it through our output guardrails.

In [ ]:
def secure_generate(prompt: str) -> str:
    print(f"[1] Prompt: {prompt}")
    
    # Generate response via litellm
    print("[2] Calling LiteLLM...")
    try:
        response = completion(
            model="claude-3-haiku-20240307", # You can change this to any supported model
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100
        )
        output_text = response.choices[0].message.content
    except Exception as e:
        return f"LiteLLM Error: {e}"
        
    print(f"[3] Model Output: {output_text}")
    
    # Run output guardrails
    print("[4] Running Output Scanners...")
    allowed, result = process_output(prompt, output_text)
    
    if not allowed:
        return f"❌ BLOCKED: {result}"
        
    return f"✅ SECURE OUTPUT: {result}"

# Test with a normal prompt
print(secure_generate("What is machine learning?"))

### Exercise 5.1: Triggering Output Guards
Try to write a prompt that forces the LLM to trigger one of our output scanners (like Toxicity or a Refusal).

In [ ]:
# TODO: Write a prompt that might cause a refusal or toxic output
tricky_prompt = ""

if tricky_prompt:
    print(secure_generate(tricky_prompt))